# 6-тарау. Автокодировщик: сығу және қалпына келтіру

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 6.1. Сығу арқылы үйрену

## 6.2. Деректер мен модель

In [ ]:
import urllib.request
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

URL = ("https://storage.googleapis.com/tensorflow/"
       "tf-keras-datasets/mnist.npz")
urllib.request.urlretrieve(URL, "mnist.npz")
d = np.load("mnist.npz")
X_train = (torch.tensor(d["x_train"], dtype=torch.float32)
           .view(-1, 784) / 255)
X_test = torch.tensor(d["x_test"], dtype=torch.float32).view(-1, 784) / 255
y_test = torch.tensor(d["y_test"])
print(X_train.shape, X_test.shape)

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, latent=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, latent))
        self.decoder = nn.Sequential(
            nn.Linear(latent, 64), nn.ReLU(),
            nn.Linear(64, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid())

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

In [ ]:
def train_ae(model, epochs=10, noise=0.0):
    dl = DataLoader(TensorDataset(X_train), batch_size=128, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(1, epochs + 1):
        total = 0.0
        for (xb,) in dl:
            inp = (xb + noise * torch.randn_like(xb)).clamp(0, 1)
            recon, _ = model(inp)
            loss = nn.functional.binary_cross_entropy(recon, xb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(xb)
        if epoch % 5 == 0:
            print(f"{epoch:2d}-дәуір: шығын={total / len(X_train):.4f}")
    return model

torch.manual_seed(0)
ae2 = train_ae(AutoEncoder(latent=2))

## 6.3. Неше сан жеткілікті?

In [ ]:
torch.manual_seed(0)
ae16 = train_ae(AutoEncoder(latent=16))

@torch.no_grad()
def test_loss(model):
    recon, _ = model(X_test)
    return nn.functional.binary_cross_entropy(recon, X_test).item()

print(f"2 сан:  {test_loss(ae2):.4f}")
print(f"16 сан: {test_loss(ae16):.4f}")
with torch.no_grad():
    rec2, _ = ae2(X_test[:8])
    rec16, _ = ae16(X_test[:8])

## 6.4. Жасырын кеңістікке үңілеміз

In [ ]:
with torch.no_grad():
    _, z = ae2(X_test)
for digit in range(10):
    m = z[y_test == digit].mean(0)
    print(digit, m.numpy().round(1))

## 6.5. Жаңа сурет жасауға бола ма?

In [ ]:
lo, hi = z.quantile(0.01, dim=0), z.quantile(0.99, dim=0)
t = torch.linspace(0, 1, 8)
grid = torch.stack(torch.meshgrid(t, t, indexing="ij"), dim=-1)
z_grid = lo + grid.view(-1, 2) * (hi - lo)     # 64 нүкте
with torch.no_grad():
    imgs = ae2.decoder(z_grid)
print(z_grid.shape, imgs.shape)

## 6.6. Шуды тазартатын автокодировщик

In [ ]:
torch.manual_seed(0)
dae = train_ae(AutoEncoder(latent=32), noise=0.5)
noisy = (X_test[:8] + 0.5 * torch.randn(8, 784)).clamp(0, 1)
with torch.no_grad():
    cleaned, _ = dae(noisy)
err_noisy = ((noisy - X_test[:8]) ** 2).mean().item()
err_clean = ((cleaned - X_test[:8]) ** 2).mean().item()
print(f"шулы: {err_noisy:.4f}, тазартылған: {err_clean:.4f}")